In [1]:
# DPP in reward function
# Imports
import re
import torch
import torch.nn.functional as F
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from src.mbe import mbe_reverse_gram

In [2]:
# Cell 3: Load and prepare GSM8K dataset
dataset = load_dataset("openai/gsm8k", "main")

def extract_gold_answer(answer_text: str) -> str:
    """Extract the numeric answer after ####."""
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        return match.group(1).strip().replace(",", "")
    return ""

def format_example(example):
    """Convert to chat-style prompt and attach gold answer."""
    example["prompt"] = [{"role": "user", "content": example["question"]}]
    example["gold_answer"] = extract_gold_answer(example["answer"])
    return example

train_dataset = dataset["train"].map(format_example)
test_dataset = dataset["test"].map(format_example)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Example prompt: {train_dataset[0]['prompt']}")
print(f"Gold answer: {train_dataset[0]['gold_answer']}")

Train: 7473, Test: 1319
Example prompt: [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]
Gold answer: 72


In [3]:
# Cell: Define reward functions

def extract_answer_from_completion(text: str) -> str:
    """Parse the final numeric answer from a model completion."""
    # Look for #### pattern first
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).strip().replace(",", "")
    # Fallback: last number in the text
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    return ""

def correctness_reward(completions: list[list[dict]], gold_answer: list[str], **kwargs) -> list[float]:
    """Award +1.0 if the model's final numeric answer matches the gold answer."""
    rewards = []
    for completion, gold in zip(completions, gold_answer):
        text = completion[0]["content"]
        predicted = extract_answer_from_completion(text)
        try:
            correct = float(predicted) == float(gold)
        except (ValueError, TypeError):
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards

def format_reward(completions: list[list[dict]], **kwargs) -> list[float]:
    """Award +0.5 if the response contains #### <number> pattern."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        has_format = bool(re.search(r"####\s*[\d,\.\-]+", text))
        rewards.append(0.5 if has_format else 0.0)
    return rewards

# Quick sanity check
test_comp = [[{"content": "The answer is 2+3=5. #### 5"}]]
print("Correctness:", correctness_reward(test_comp, gold_answer=["5"]))
print("Format:", format_reward(test_comp))

Correctness: [1.0]
Format: [0.5]


## Intuitions of DPP-Enhanced GRPO

- Penalize the group if the outputs are too similar to one another in the embedding space.
- Effectively, we want the Determinant of the group's similarity matrix to be high, as the determinant represents the volume spanned by the vectors in feature space which equates to more diversity.

### Formulation
Let $L$ be a $G \times G$ positive semi-definite matrix where the entry $L_{ij}$ represents the similarity between output $i$ and output $j$. The standard decomposition for a DPP kernel $L$ is: $$L_{ij} = q_i \cdot \phi(o_i)^\top \phi(o_j) \cdot q_j$$
- $q_i = \exp(\text{reward}(o_i))$ is the quality score. 
- $S_{ij} = \phi(o_i)^\top \phi(o_j)$ is the similarity score.

The probability of selecting a specific set of outputs under a DPP is proportional to $\det(L)$.

<!-- To bake this into the GRPO objective, we add a DPP regularization term to the reward function: $$R_{DPP}(o_i) = R_{0}(o_i) + \alpha \cdot \frac{\log \det(L)}{G}$$
- $R_{0}$ denotes the standard reward.
- $\alpha$: A hyperparameter controlling the strength of the diversity push. -->

Using the log-det makes the gradient more stable: 
$$\log \det(L) = \log \det(\text{diag}(\mathbf{q}) \cdot S \cdot \text{diag}(\mathbf{q})) = \sum_{i=1}^{G} 2 \log(q_i) + \log \det(S)$$

The marginal gain in diversity for an individual sample $o_i$ is given by its gains vs leaving it out from the group: $$\text{Reward}_{diversity}(o_i) = \log \det(L) - \log \det(L_{-i})$$

In [4]:
@torch.no_grad()
def full_forward(model, input_ids):
    """Single forward pass returning logits and all hidden states."""
    outputs = model(input_ids, output_hidden_states=True, use_cache=False)
    return outputs.logits, outputs.hidden_states

def compute_log_det(embeddings, jitter=1e-4):
    """Computes log(det(S)) where S is the cosine similarity matrix."""
    G = embeddings.size(0)
    # Normalize for cosine similarity
    norm_emb = F.normalize(embeddings, p=2, dim=1)
    S = torch.mm(norm_emb, norm_emb.t())
    # Add jitter for numerical stability (ensures positive definiteness)
    S = S + torch.eye(G, device=S.device) * jitter
    # Use Cholesky for stable log-det: log|S| = 2 * sum(log(diag(L)))
    L_chol = torch.linalg.cholesky(S)
    return 2 * torch.sum(torch.log(torch.diagonal(L_chol)))

class DPPReward:
    def __init__(self, tokenizer, layer=-1, alpha=1.0):
        self.__name__ = "dpp_reward"
        self.model = None
        self.tokenizer = tokenizer
        self.layer = layer
        self.alpha = alpha # this is the weight of the DPP reward

    def set_model(self, model):
        self.model = model

    @torch.no_grad()
    def __call__(self, prompts, completions, **kwargs) -> list[float]:
        if self.model is None or len(completions) <= 1:
            return [0.0] * len(completions)

        rewards = []
        device = next(self.model.parameters()).device
        
        # 1. Extract Embeddings for the entire group
        # In GRPO, prompts/completions are delivered in groups for the same prompt
        group_embeddings = []
        for prompt, completion in zip(prompts, completions):
            prompt_text = self.tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            full_text = prompt_text + completion[0]["content"]
            
            inputs = self.tokenizer(full_text, return_tensors="pt").to(device)
            outputs = self.model(inputs.input_ids, output_hidden_states=True, use_cache=False)
            
            # Use the last token's hidden state as the "semantic vector"
            # (Batch=1, Seq, Dim) -> (Dim)
            emb = outputs.hidden_states[self.layer][0, -1, :]
            a = outputs.hidden_states[self.layer]
            group_embeddings.append(emb)
        # Stack into (G, D)
        all_embs = torch.stack(group_embeddings)
        G = all_embs.size(0)

        # 2. Calculate Total Group Log-Determinant
        total_log_det = compute_log_det(all_embs)

        # 3. Calculate Leave-One-Out Marginal Gain
        # Reward_i = alpha * (LogDet(Full) - LogDet(Full without i))
        for i in range(G):
            # Mask out index i
            mask = torch.arange(G, device=device) != i
            reduced_embs = all_embs[mask]
            
            partial_log_det = compute_log_det(reduced_embs)
            
            # Marginal gain: how much diversity does this specific sample add?
            marginal_gain = (total_log_det - partial_log_det).item()
            rewards.append(self.alpha * marginal_gain)
            
        return rewards

In [ ]:
# Cell: Training with DPP reward
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# DPP reward (deferred model binding — trainer loads the model)
dpp_reward = DPPReward(tokenizer)

config = GRPOConfig(
    output_dir="dpp_reward_gsm8k_output",
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=10,
    max_steps=20,
    use_vllm=False,
    # vllm_mode="colocate",
    # vllm_gpu_memory_utilization=0.3,
    bf16=True,
    save_strategy="no",
    report_to="none",
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[correctness_reward, format_reward, dpp_reward],
    args=config,
    train_dataset=train_dataset,
)

# Bind model ref for DPP computation after trainer has loaded it
dpp_reward.set_model(trainer.model)

trainer.train()